### 1~3 각 계별 jupyter note에서 전처리 후, pickle 또는 csv저장.

1. 전국일방통행도로표준데이터
    - 시도명, 지정사유, 지정연도, 도로폭, 도로차로수, 보차분리여부
    
    
2. 전국어린이보호구역표준데이터 시도명 작업 필요 (제공기관명에서 split)
    - 시설종류, 관할경찰서명, CCTV설치여부, CCTV설치대수, 제공기관명, 보호구역도로폭
    
    
3. 전국노인장애인보호구역 표준데이터
    - 시도명, 제한 속도,  CCTV설치여부, CCTV설치대수, 보호구역도로폭
    
    
4. 새로운 jupyter note에서 종합
    - 시도별, 어린이보호구역 CCTV 설치대수, 노인장애인보호구역 CCTV 설치 대수, 일방통행 보차분리 ycount

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def check_nan(pd_data):
    for e in pd_data:
        print(e,'\t',pd_data[e].hasnans)

In [3]:
# 1. 전국일방통행도로표준데이터
#     - 시도명, 지정사유, 지정연도, 도로폭, 도로차로수, 보차분리여부

pd_raw_OW = pd.read_csv('전국일방통행도로표준데이터.csv', encoding='cp949')

col_selection = ['시도명', '지정사유','지정연도','도로폭','도로차로수','보차분리여부']
pd_data_OW = pd_raw_OW[col_selection]

def myfn1(pd_raw, pd_data):
    filter1 = pd_raw['시도명']=='10'
    pd_data.loc[filter1,'시도명'] = '강원도'
    return pd_data

# print(myfn1(pd_data_OW))
# print(pd_data_OW['시도명'].unique())
# print(pd_data_OW['시도명'].hasnans)
def e1(x):
    if '원활' in x:
        return '원활'
    elif '불편' in x:
        return '불편'
    elif '안전' in x:
        return '안전'
    elif '혼잡' in x:
        return '혼잡'
    elif '혼자지역' in x:
        return '혼잡'
    elif '주민' in x:
        return '주민'
    elif '교행' in x:
        return '교행'
    elif '도로' in x:
        return '도로진입'
    elif '출입구' in x:
        return '출입구'
    elif '교통사고' in x:
        return '교통사고'
    else: 
        return x

def myfn2(pd_raw, pd_data):
    na_filter = pd_data['지정사유'].isna()
    pd_data.loc[na_filter, '지정사유'] = '불분명'
    return pd_data

# print(pd_data_OW['지정사유'].unique())
# print(pd_data_OW['지정사유'].hasnans)

def myfn3(pd_raw, pd_data):
    na_filter = pd_data['지정연도'].isna()
    pd_data.loc[na_filter, '지정연도'] = 0
    return pd_data
 
# print(pd_data_OW['지정연도'].unique())
# print(pd_data_OW['지정연도'].hasnans)

def myfn4(pd_data):
    q1, q3 = pd_data['도로폭'].quantile([0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 1.5*iqr
    lower = q1 + 1.5*iqr
#     print(upper, lower)
    filter1 = pd_data['도로폭']>upper
    pd_data.loc[filter1, '도로폭'] = np.NaN
    return pd_data

# print(pd_data_OW['도로폭'].unique())
# print(pd_data_OW['도로폭'].hasnans)

def myfn5(pd_data):
# 결측치: 1개 샘플, 후에 dropna()이용 제거
    na_filter = pd_data['도로차로수'].isna()
# 오류치: 60 --> np.NaN로 대체, 후에 dropna()이용 제거
    filter1 = pd_data['도로차로수']==60.0
    pd_data.loc[filter1, '도로차로수']=np.NaN
    return pd_data

# print(pd_data_OW['도로차로수'].unique())
# print(pd_data_OW['도로차로수'].hasnans)

def myfn6(pd_data):
    na_filter = pd_data['보차분리여부'] == ' '
    pd_data.loc[na_filter, '보차분리여부']=np.NaN
    return pd_data

# print(pd_data_OW['보차분리여부'].unique())
# print(pd_data_OW['보차분리여부'].hasnans)


pd_data_OW = myfn1(pd_raw_OW ,pd_data_OW)

pd_data_OW = myfn2(pd_raw_OW ,pd_data_OW)
pd_data_OW.loc[:,'지정사유'] = pd_data_OW.loc[:,'지정사유'].apply(e1)
pd_data_OW = myfn3(pd_raw_OW ,pd_data_OW)
pd_data_OW = myfn4(pd_data_OW)
pd_data_OW = myfn5(pd_data_OW)
pd_data_OW = myfn6(pd_data_OW)

pd_data_ow_t = pd_data_OW.dropna()
pd_data_ow_t['지정사유'].unique()

check_nan(pd_data_ow_t)

시도명 	 False
지정사유 	 False
지정연도 	 False
도로폭 	 False
도로차로수 	 False
보차분리여부 	 False


C:\Users\ITPS\AppData\Local\Temp\ipykernel_4276\703174875.py:93: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_data_OW.loc[:,'지정사유'] = pd_data_OW.loc[:,'지정사유'].apply(e1)


In [24]:
# 2. 전국어린이보호구역표준데이터 시도명 작업 필요 (제공기관명에서 split)
#     - 시설종류, 관할경찰서명, CCTV설치여부, CCTV설치대수, 제공기관명, 보호구역도로폭

pd_raw = pd.read_csv('전국어린이보호구역표준데이터.csv', encoding='cp949')

col_selection = ['시설종류', '관할경찰서명','CCTV설치여부','CCTV설치대수','제공기관명','보호구역도로폭']
pd_data_CH = pd_raw[col_selection]
# print(pd_data_CH.dtypes)

pd_data_CH['시설종류'] = pd_data_CH['시설종류'].astype('category')
pd_data_CH['CCTV설치여부'] = pd_data_CH['CCTV설치여부'].astype('category')
pd_data_CH['제공기관명'] = pd_data_CH['제공기관명'].astype('category')

# check_nan(pd_data_CH)
print(pd_data_CH.dtypes)

시설종류        category
관할경찰서명        object
CCTV설치여부    category
CCTV설치대수     float64
제공기관명       category
보호구역도로폭       object
dtype: object


C:\Users\ITPS\AppData\Local\Temp\ipykernel_4276\666474850.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_data_CH['시설종류'] = pd_data_CH['시설종류'].astype('category')
C:\Users\ITPS\AppData\Local\Temp\ipykernel_4276\666474850.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_data_CH['CCTV설치여부'] = pd_data_CH['CCTV설치여부'].astype('category')
C:\Users\ITPS\AppData\Local\Temp\ipykernel_4276\666474850.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try usi

In [36]:
# 시설종류 결측치 없음.
# print(pd_data_CH['시설종류'].unique())
# print(pd_data_CH['시설종류'].hasnans)

# 관할경찰서 결측치 없음, 오류치는 나중에 판단.
# print(pd_data_CH['관할경찰서명'].unique())
# print(pd_data_CH['관할경찰서명'].hasnans)

#CCTV설치여부 결측치 오류치 없음
# print(pd_data_CH['CCTV설치여부'].unique())
# print(pd_data_CH['CCTV설치여부'].hasnans)

#CCTV설치대수 결측치 있음, 설치 0 있음 --> 1 또는 평균, nan에 설치여부에 따라 설치 대수 조정
def yn_filter(pd_data):
    filter_y = pd_data['CCTV설치여부'] == 'Y'
    filter_n = pd_data['CCTV설치여부'] == 'N'
    pd_data.loc[filter_y, 'CCTV설치대수'] = pd_data.loc[filter_y, 'CCTV설치대수'].replace(np.NaN,1)
    pd_data.loc[filter_n, 'CCTV설치대수'] = pd_data.loc[filter_n, 'CCTV설치대수'].replace(np.NaN,0)
    return pd_data
# print(pd_data_CH['CCTV설치대수'].unique())
# print(pd_data_CH['CCTV설치대수'].hasnans)

#제공기관명 도까지 split 하여서 시도명과 붙일것
# print(pd_data_CH['제공기관명'].unique())
# print(pd_data_CH['제공기관명'].hasnans)
def myfn7(pd_data):
    if ' ' in pd_data['제공기관명']:
        return e[:e[e.index(' ')]]


# 보호구역도로폭 nan 결측치는 평균, 0 --> 평균 , ~는 앞뒤 숫자의 평균으로 
# print(pd_data_CH['보호구역도로폭'].unique())
# print(pd_data_CH['보호구역도로폭'].hasnans)
def myfn8(x):
    if type(x) == type(' '):
        if '~' in x:
            m = np.array(x.split('~')).astype(np.float64).mean()
            return str(m)

In [26]:
pd_data_CH = yn_filter(pd_data_CH)
print(pd_data_CH['CCTV설치대수'].unique())
print(pd_data_CH['CCTV설치대수'].hasnans)

[ 1.  2.  3.  5.  4.  6.  0.  7. 24. 10.  9. 14.  8. 11. 20. 19. 18. 15.
 12. 34. 13. 36. 17. 23. 46. 21. 16. 22. 28. 26. 29. 32. 35.]
False


In [37]:
pd_data_CH = myfn7(pd_data_CH)
print(pd_data_CH['제공기관명'].unique())
print(pd_data_CH['제공기관명'].hasnans)

TypeError: 'NoneType' object is not subscriptable

In [21]:
y = pd_data_CH['보호구역도로폭'].apply(myfn8)
# print(pd_data['보호구역도로폭'].isna().value_counts())
y = y.astype(np.float64)
y = y.replace(np.NaN, y.mean())
# print(y.isna().value_counts())
pd_data_CH['보호구역도로폭'] = y
print(pd_data_CH['보호구역도로폭'].unique())
print(pd_data_CH['보호구역도로폭'].hasnans)

TypeError: string indices must be integers

In [4]:
# 3. 전국노인장애인보호구역 표준데이터
#     - 시도명, 제한속도,  CCTV설치여부, CCTV설치대수, 보호구역도로폭

pd_raw = pd.read_csv('전국노인장애인보호구역표준데이터.csv', encoding='cp949')

col_selection = ['시도명', '제한속도','CCTV설치여부','CCTV설치대수','보호구역도로폭']
pd_data_SE = pd_raw[col_selection]
print(pd_data_SE)

        시도명  제한속도 CCTV설치여부  CCTV설치대수 보호구역도로폭
0     서울특별시    30        Y       1.0       3
1      경상북도    30        N       NaN       6
2      경상북도    30        N       NaN       6
3      경상북도    30        N       NaN       7
4      경상북도    30        N       NaN       7
...     ...   ...      ...       ...     ...
2513  대구광역시    30        N       NaN     NaN
2514  대구광역시    30        N       NaN     NaN
2515  대구광역시    30        N       NaN     NaN
2516  대구광역시    30        N       NaN     NaN
2517  대구광역시    30        N       NaN     NaN

[2518 rows x 5 columns]
